# VM PF15 — Can, 데모 prefill + 고정 α 0.15: `can_prefill_fixa015` × 3 (vm2_new 판형, 0→9 순서)

전용 G4 VM에서 **0부터 9까지 순서대로**. 1번은 런타임 재시작 → 2번부터. 코드 변경 없음, 3 run(RAM ≈ 51 GB, ~3시간), 9번이 끝나면 VM 반납. 기준 문서: `HANDOFF.md` §23, `results/2026-09-12/README.md`, 그림 `results/2026-09-12/prefill_alpha/`.

**앞 VM(9/12 PA)의 결과**: `can_prefill_fixa03`(α 0.3 고정)은 5k seed 평균 0.493 통과, 10k/15k 0.500/0.480(mix_prefill형 dip 없음), 그러나 fixalpha_03 대비 어느 시점도 이득이 없고(초기 AUC −0.025 ± 0.030, 104k/129k 평균 −0.004 ± 0.047) 후반 0.736 ± 0.030으로 auto-α 데모 arm(prefill_t12i 0.867 ± 0.036, a015 0.862 ± 0.010)보다 −0.13. `can_prefill_t12i_a015`(auto-α 0.15 시작)는 5k 0.283 실패 — α 손실 기울기가 H − 목표라 H₀ 16–19 > 12인 한 **초기값과 무관하게 α는 먼저 떨어져**(0.15 → 0.075@2.4k) 과도가 재현됐고, 회복만 빨랐다(T80 15k). 즉 PA는 "5k dip = α/엔트로피 **수준** vs **과도**"와 "데모 후반 이득 = α **수준** vs **제어기**"를 가르지 못했다.

## 질문: 제어기 없이 α를 0.15에 고정하고 데모를 prefill하면 (1) 5k에 dip이 있는가, (2) 후반이 auto-α 데모 arm(~0.86)인가 고정 0.3 arm(~0.74)인가
- **단일 arm `can_prefill_fixa015_s{1,2,3}`**: `offline_mix.mode=prefill … train.ent_coef=0.15`(고정, `target_ent` 없음). 과도가 없으므로 5k 시점의 α는 0.150 그대로이고, 엔트로피는 fixalpha_01(5k 행 H 7.3–7.7, |μ| 0.60)과 fixalpha_03(H 13.4–15.6, |μ| 0.33–0.43) 사이일 것으로 예상.
- 비교군(모두 있음): `can_prefill_fixa03`(같은 데모, α 0.3), `can_prefill_t12i`·`can_prefill_t12i_a015`(같은 데모, auto-α, 5k 행 α 0.10–0.12), `can_fixalpha_03`·`can_fixalpha_01`(데모 없음, 고정 α), `can_mix_prefill`.

## 사전 판정 (§22·§23과 같은 정의; 최저 하나로 판정하지 않는다)
- **주 지표 1 = online 5,008의 seed 평균** ≥ 0.405 (참조: prefill_fixa03 0.493, fixalpha_03 s1–3 0.443, prefill_t12i_a015 0.283, prefill_t12i 0.097, mix_prefill 0.780, fixalpha_01 0.143) + seed-matched 차이.
  - ≥ 0.405 → 5k dip은 **제어기 과도**(α 0.075 undershoot, H 8.6–10.5 골)에 묶인 것이지 α≈0.15 수준만으로는 안 생긴다.
  - < 0.405 → **수준**: 과도 없이도 α 0.15(와 그때의 H)면 dip. fixalpha_01(3/3 dip)과 같은 편.
  - 단, 이 읽기는 5k 직전 행의 상태가 α 0.150 / H 9–13 밴드에 있을 때만 성립한다. H가 fixalpha_01처럼 7 근처면 "수준" 쪽 증거는 α가 아니라 엔트로피의 몫이므로 그렇게 쓴다.
- **주 지표 2 = 104k/129k 평균**(env 104,144·129,152, 각 200 ep). 참조 prefill_fixa03 0.736 ± 0.030 vs prefill_t12i 0.867 ± 0.036 / a015 0.862 ± 0.010.
  - ≥ 0.80 → 데모의 후반 이득은 제어기 없이도 나온다 → 고정 0.3의 무익함은 **α 수준(0.3이 너무 높음)**의 문제.
  - ≤ 0.78 → 고정 α로는 α 수준을 낮춰도 못 얻는다 → **제어기/엔트로피 궤적**의 문제.
  - 0.78–0.80 → 미정. seed-matched: vs prefill_fixa03(수준이면 ≈ +0.13), vs a015(수준이면 ≈ 0).
- 보조: 10k·15k seed 평균(mix_prefill형 dip 0.093/0.303 유무), 초기 AUC(참조 prefill_fixa03 0.654, prefill_t12i 0.776, fixalpha_01 0.501), T80, 그리고 train_log의 5k 직전 행(env 28,816)·50k·100k의 α/H/|μ|(고정 0.15가 엔트로피를 붙드는지 fixalpha_01처럼 15k까지 H 3–5로 무너뜨리는지 — 판정 기준은 아니고 해석 조건).
- 검정력 주의(§23): 3-seed 5k 평균의 SE ≈ 0.10–0.17이고 fixalpha_03·baseline의 3-seed 부분집합도 8/10이 0.405를 넘는다. 5k 판정은 prefill_t12i형(0.097) dip 유무를 가를 뿐 fixalpha_03과의 구분은 아니다. 후반 판정은 참조 두 군이 2.8–4.0 SE 떨어져 있어 n=3으로도 어느 편인지는 읽힌다.
- 쓰면 안 되는 문장은 `HANDOFF.md` §23 목록 그대로("α 0.15가 답", "dip 제거", 단조 사다리, Q_W 부호 반전 표지, 붕괴 상태 원인).

평가 격자는 기존과 같은 5k(비교 가능성). 이 arm 뒤에는 추가 α 사다리를 돌리지 않는다(§23 '하지 않는 것').


## 0. Drive 마운트와 GPU 확인

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Conda 설치 — 실행하면 런타임 자동 재시작

In [ ]:
!pip install -q condacolab
import condacolab
condacolab.install()

## 2. 재시작 후 Drive 재마운트

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
PROJ = '/content/drive/MyDrive/dsrl_project'
print(PROJ)

## 3. 최신 o2o 저장소 준비

In [ ]:
%%bash
set -e
git config --global url."https://github.com/".insteadOf "git@github.com:"
if [ -d /content/dsrl/.git ]; then
  git -C /content/dsrl checkout o2o
  git -C /content/dsrl pull --ff-only origin o2o
else
  test ! -e /content/dsrl || { echo '/content/dsrl exists but is not a git checkout'; exit 2; }
  git clone --recurse-submodules -b o2o https://github.com/msp0617/dsrl.git /content/dsrl
fi
git -C /content/dsrl submodule sync --recursive
git -C /content/dsrl submodule update --init --recursive
echo -n 'HEAD: '; git -C /content/dsrl rev-parse --short HEAD

## 4. 캐시에서 conda 환경 복원

In [ ]:
%%bash
set -e
CACHE=/content/drive/MyDrive/dsrl_project/env_cache/dsrl_env.tar.gz
test -s "$CACHE" || { echo "missing $CACHE"; exit 2; }
mkdir -p /usr/local/envs
rm -rf /usr/local/envs/dsrl
tar -xzf "$CACHE" -C /usr/local/envs
source /usr/local/etc/profile.d/conda.sh
conda activate dsrl
python - <<'PY'
import torch, robomimic, robosuite, mujoco, stable_baselines3
assert torch.cuda.is_available(), 'GPU runtime required'
print('torch', torch.__version__, '| GPU', torch.cuda.get_device_name(0))
PY
python /usr/local/envs/dsrl/lib/python3.10/site-packages/robosuite/scripts/setup_macros.py

## 5. Can 정책·정규화 복원과 환경 패치

In [ ]:
%%bash
set -e
source /usr/local/etc/profile.d/conda.sh && conda activate dsrl
PROJ=/content/drive/MyDrive/dsrl_project
RUNTIME=/content/dsrl/dppo/log
DRIVE=$PROJ/dppo_log
mkdir -p "$RUNTIME"
test -d "$DRIVE" || { echo "missing $DRIVE"; exit 2; }
cp -r "$DRIVE"/. "$RUNTIME"/
CKPT_REL=robomimic-pretrain/can/can_pre_diffusion_mlp_ta4_td20/2024-06-28_13-29-54/checkpoint/state_5000.pt
NORM_REL=robomimic/can/normalization.npz
CKPT_SRC=$(find "$RUNTIME" "$DRIVE" -type f -path "*/$CKPT_REL" -print -quit)
NORM_SRC=$(find "$RUNTIME" "$DRIVE" -type f -path "*/$NORM_REL" -print -quit)
test -n "$CKPT_SRC" -a -n "$NORM_SRC" || { echo 'Can policy assets missing'; exit 2; }
mkdir -p "$RUNTIME/$(dirname "$CKPT_REL")" "$RUNTIME/$(dirname "$NORM_REL")"
[ "$CKPT_SRC" = "$RUNTIME/$CKPT_REL" ] || cp -f "$CKPT_SRC" "$RUNTIME/$CKPT_REL"
[ "$NORM_SRC" = "$RUNTIME/$NORM_REL" ] || cp -f "$NORM_SRC" "$RUNTIME/$NORM_REL"
cat > /content/env.sh <<'EOS'
export MUJOCO_GL=egl
export PYOPENGL_PLATFORM=egl
export WANDB_MODE=disabled
EOS
python /content/dsrl/colab/patch_env.py
ls -lh "$RUNTIME/$CKPT_REL" "$RUNTIME/$NORM_REL"

## 6. 사전검사 — 데모 npz, 이번 exp_id 3개가 비어 있는지(있으면 같은 명령이 checkpoint resume), 비교군, RAM·디스크


In [ ]:
%%bash
set -e
source /usr/local/etc/profile.d/conda.sh && conda activate dsrl
PROJ=/content/drive/MyDrive/dsrl_project
python - <<'PY'
import numpy as np
path = '/content/drive/MyDrive/dsrl_project/offline/can_train_offline.npz'
with np.load(path) as data:
    assert len(data['states']) > 0
    print('OK', path.split('/')[-1], 'rows', len(data['states']))
PY
for E in can_prefill_fixa015_s1 can_prefill_fixa015_s2 can_prefill_fixa015_s3; do
  if [ -f "$PROJ/logs/$E.out" ]; then echo "$E: 이미 있음 -> $(grep '\[done\]\|\[eval\]' $PROJ/logs/$E.out | tail -n 1 | cut -c1-80) (같은 명령이면 resume)"; else echo "$E: 새로 시작"; fi
done
for G in prefill_fixa03 prefill_t12i prefill_t12i_a015 mix_prefill fixalpha_03 fixalpha_01; do echo -n "비교군 can_$G: "; ls -d $PROJ/logs/can_${G}_s* 2>/dev/null | wc -l; done
echo "processes: $(pgrep -fc '[t]rain_dsrl.py' || true)"; free -g | head -2; df -h /content | tail -n 1


## 7. 3개 시작 — `can_prefill_fixa015` × 3 (150k, 5k 격자, `train.ent_coef=0.15` 고정, `target_ent` 없음)


In [ ]:
%%bash
set -e
source /usr/local/etc/profile.d/conda.sh && conda activate dsrl
source /content/env.sh
cd /content/dsrl
git pull --ff-only origin o2o
PROJ=/content/drive/MyDrive/dsrl_project
mkdir -p "$PROJ/logs"
CFG='--config-path=cfg/robomimic --config-name=dsrl_can.yaml'
PREFILL="offline_mix.mode=prefill offline_data_path=$PROJ/offline/can_train_offline.npz"
launch () {
  EXP=$1; shift
  if pgrep -af '[t]rain_dsrl.py' | grep -Fq "exp_id=$EXP"; then echo "already running: $EXP"; return; fi
  nohup python train_dsrl.py $CFG exp_id=$EXP "$@" > "$PROJ/logs/$EXP.out" 2>&1 &
  echo "started $EXP (pid $!)"
}
for S in 1 2 3; do
  launch can_prefill_fixa015_s$S seed=$S variant=baseline log_dir=$PROJ/logs train.total_env_steps=150000 $PREFILL train.ent_coef=0.15
done


## 8. 3분 후 자동 확인 — 3개 running, ERR 없음, 인자에 `train.ent_coef=0.15`와 `offline_mix.mode=prefill`(`target_ent` 없음). 20~30분 뒤 다시 돌리면 train_log의 α(고정 0.150)·logp·offline_p(≈0.9)도 찍힌다


In [ ]:
%%bash
sleep 180
source /usr/local/etc/profile.d/conda.sh && conda activate dsrl
cd /content/dsrl
PROJ=/content/drive/MyDrive/dsrl_project
python scripts/inspect_runs.py --proj "$PROJ" --only can_prefill_fixa015_s
for E in can_prefill_fixa015_s1 can_prefill_fixa015_s2 can_prefill_fixa015_s3; do
  echo "== $E: $(grep '\[budget\]\|\[eval\]\|Traceback\|Error' "$PROJ/logs/$E.out" | tail -n 2 | tr '\n' ' ' | cut -c1-160)"
  T=$PROJ/logs/$E/train_log.csv
  [ -f "$T" ] && awk -F, 'NR==1{for(i=1;i<=NF;i++)c[$i]=i} END{printf "   train_log last: env_steps=%s ent_coef=%s logp_mean=%s qw_mean=%s mu=%s offline_p=%s\n", $c["env_steps"], $c["ent_coef"], $c["logp_mean"], $c["qw_mean"], $c["mu_absmean"], $c["offline_p"]}' "$T"
done
echo '== processes (인자 확인)'; pgrep -af '[t]rain_dsrl.py' | sed 's/.*exp_id=/exp_id=/' | cut -c1-200 || true
free -g | head -2


## 9. Keepalive — 마지막 프로세스가 끝나면 자동 반납

8번에서 오류가 없을 때만 실행하고 이 셀을 계속 실행 상태로 둡니다.

In [ ]:
import subprocess, time
from pathlib import Path
EXPECTED = [f'can_prefill_fixa015_s{s}' for s in (1, 2, 3)]
LOGS = Path('/content/drive/MyDrive/dsrl_project/logs')
assert LOGS.is_dir(), 'Drive가 이 런타임에 마운트돼 있지 않음 — 0번(또는 2번) 셀 먼저'

def running():
    out = subprocess.run(['ps', '-eo', 'pid,args'], capture_output=True, text=True).stdout
    return [line.strip() for line in out.splitlines() if 'train_dsrl.py' in line and any(f'exp_id={e}' in line for e in EXPECTED)]

def last_event(exp):
    path = LOGS / f'{exp}.out'
    if not path.exists(): return 'NO .out'
    lines = path.read_text(errors='replace').splitlines()[-500:]
    for line in reversed(lines):
        if any(x in line for x in ('[eval]', '[done]', 'Traceback', 'Error')): return line[:100]
    return 'starting'

seen_running = False
while True:
    procs = running()
    events = {e: last_event(e) for e in EXPECTED}
    print(time.strftime('%H:%M'), f'running {len(procs)}/{len(EXPECTED)}', '|', ' | '.join(f'{e}: {v}' for e, v in events.items()), flush=True)
    if procs:
        seen_running = True
    elif seen_running or all(v.startswith('[done]') for v in events.values()):
        print('all VM PF15 runs stopped -> unassigning', flush=True)
        from google.colab import runtime
        runtime.unassign()
        break
    else:
        print('이 런타임에 실행 중인 run이 없고 [done]도 아님 -> 7번 셀이 안 돌았거나 다른 런타임입니다. 반납하지 않고 종료.', flush=True)
        break
    time.sleep(600)


## 10. 결과 zip (끝난 뒤, CPU 런타임 + 0번 Drive 마운트만으로 됨). 로컬에서는 **새 폴더**(예: `logs/bundle_<날짜>/`)에 풀고 — 옛 번들이 섞인 `~/Downloads/logs`는 쓰지 않는다 —
`python scripts/plot_results.py --logs <새폴더>/logs --out results/<날짜>/prefill_fixa015 --axes "prefill_fixa015=baseline,mix_prefill,prefill_t12i,prefill_fixa03,prefill_t12i_a015,prefill_fixa015;alpha_level=fixalpha_01,fixalpha_03,prefill_fixa03,prefill_fixa015"`


In [ ]:
%%bash
cd /content/drive/MyDrive/dsrl_project
rm -f csv_bundle.zip
zip -qr csv_bundle.zip logs -i "logs/*/eval_log.csv" "logs/*/train_log.csv" "logs/*.csv" "logs/pretrain/*_log.csv"
ls -lh csv_bundle.zip